# 01 - Feature Extraction

Notebook này chỉ làm một việc: trích xuất và lưu feature `.npy` cho STL-10 và BloodMNIST. Clustering notebooks về sau chỉ đọc các file `.npy` này.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

try:
    import medmnist
    from medmnist import INFO
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'medmnist'])
    import medmnist
    from medmnist import INFO

sys.path.append('..')
from src.utils import get_device, set_seed

set_seed(42)
device = get_device()
DATA_DIR = Path('../data')
CKPT_DIR = DATA_DIR / 'checkpoints'
DATA_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)
print(f'[INFO] device = {device}')


In [ ]:
MOCO_URL = 'https://dl.fbaipublicfiles.com/moco/moco_checkpoints/moco_v2_800ep/moco_v2_800ep_pretrain.pth.tar'
MOCO_CKPT = CKPT_DIR / 'moco_v2_800ep_pretrain.pth.tar'

if not MOCO_CKPT.exists():
    print('[INFO] Downloading official MoCo-v2 checkpoint...')
    torch.hub.download_url_to_file(MOCO_URL, str(MOCO_CKPT), progress=True)
else:
    print(f'[INFO] Reusing cached checkpoint: {MOCO_CKPT}')

print(MOCO_CKPT)


In [ ]:
def build_moco_v2_resnet50_feature_extractor(checkpoint_path, device):
    backbone = torchvision.models.resnet50(weights=None)
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    state_dict = checkpoint['state_dict'] if 'state_dict' in checkpoint else checkpoint

    cleaned_state = {}
    for key, value in state_dict.items():
        if not key.startswith('module.encoder_q.'):
            continue
        new_key = key[len('module.encoder_q.'): ]
        if new_key.startswith('fc.'):
            continue
        cleaned_state[new_key] = value

    missing, unexpected = backbone.load_state_dict(cleaned_state, strict=False)
    print('[INFO] Missing keys:', missing)
    print('[INFO] Unexpected keys:', unexpected)

    feature_extractor = nn.Sequential(*list(backbone.children())[:-1]).to(device)
    feature_extractor.eval()
    return feature_extractor


def build_resnet18_feature_extractor(device):
    model = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
    feature_extractor = nn.Sequential(*list(model.children())[:-1]).to(device)
    feature_extractor.eval()
    return feature_extractor


def extract_and_save_features(dataset, backbone, feature_path, label_path, batch_size=128):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    features, labels = [], []
    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device)
            z = backbone(images).flatten(1)
            features.append(z.cpu().numpy())
            labels.append(torch.as_tensor(targets).view(-1).cpu().numpy())

    features = np.concatenate(features, axis=0).astype(np.float32)
    labels = np.concatenate(labels, axis=0).astype(np.int64)
    np.save(feature_path, features)
    np.save(label_path, labels)
    print(f'[INFO] Saved features to {feature_path} with shape {features.shape}')
    print(f'[INFO] Saved labels to {label_path} with shape {labels.shape}')


In [ ]:
# Task A: STL-10 -> MoCo-v2 ResNet50 -> .npy
stl_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

stl10_test = torchvision.datasets.STL10(root=str(DATA_DIR), split='test', download=True, transform=stl_transform)
moco_backbone = build_moco_v2_resnet50_feature_extractor(str(MOCO_CKPT), device)
extract_and_save_features(
    stl10_test,
    moco_backbone,
    DATA_DIR / 'stl10_moco_features.npy',
    DATA_DIR / 'stl10_labels.npy',
)


In [ ]:
# Task B: BloodMNIST -> ResNet18 -> .npy
data_flag = 'bloodmnist'
info = INFO[data_flag]
DataClass = getattr(medmnist, info['python_class'])

blood_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.repeat(3, 1, 1) if x.shape[0] == 1 else x),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

blood_test = DataClass(split='test', transform=blood_transform, download=True)
blood_backbone = build_resnet18_feature_extractor(device)
extract_and_save_features(
    blood_test,
    blood_backbone,
    DATA_DIR / 'bloodmnist_resnet_features.npy',
    DATA_DIR / 'bloodmnist_labels.npy',
)


In [ ]:
# Quick verification of saved arrays
for name in [
    'stl10_moco_features.npy',
    'stl10_labels.npy',
    'bloodmnist_resnet_features.npy',
    'bloodmnist_labels.npy',
]:
    arr = np.load(DATA_DIR / name)
    print(name, arr.shape, arr.dtype)
